In [ ]:
import subprocess
import glob
import xml.etree.ElementTree as ET
import pandas as pd
import os
import time
from pathlib import Path
import csv

XML_DIR = "raw_data_debates"
os.makedirs(XML_DIR, exist_ok=True)

def download(date):
    subprocess.run([
        "rsync", "-az",
        f"data.theyworkforyou.com::parldata/scrapedxml/debates/debates{date}*.xml",
        XML_DIR + "/"
    ], capture_output=True) # suppress rsync output

def parse(date):
    rows = []
    for filepath in sorted(glob.glob(f"{XML_DIR}/debates{date}*.xml")):
        tree = ET.parse(filepath)
        section = ""
        for elem in tree.getroot().iter():
            if elem.tag in ("major-heading", "minor-heading"):
                section = (elem.text or "").strip()
            elif elem.tag == "speech":
                text = " ".join(p.text or "" for p in elem.findall("p")).strip()
                if text:
                    rows.append({
                        "date":       date,
                        "file":       filepath,
                        "speech_id":  elem.get("id", ""),
                        "type":       elem.get("type", ""),
                        "speaker":    elem.get("speakername", ""),
                        "speaker_id": elem.get("person_id", ""),
                        "colnum":     elem.get("colnum", ""),
                        "time":       elem.get("time", ""),
                        "url":        elem.get("url", ""),
                        "section":    section,
                        "text":       text,
                        "word_count": len(text.split()),
                    })
    return rows

In [ ]:
# Download raw files
#all_rows = []
DATES = pd.date_range("1952-01-01", "2025-12-31").strftime("%Y-%m-%d").tolist()

for i, date in enumerate(DATES):
    # skip if already downloaded
    if not glob.glob(f"{XML_DIR}/debates{date}*.xml"):
        download(date)
        time.sleep(0.1)

    rows = parse(date)
    #all_rows.extend(rows)

    if rows:
        print(f"[{i+1}/{len(DATES)}] {date} — {len(rows)} speeches")

#df = pd.DataFrame(all_rows)


In [ ]:
# Create CSV file
DATES = sorted(set(
    p.stem[7:17]
    for p in Path(XML_DIR).glob("debates*.xml")
))

OUTPUT = "hansard.csv"

# write header once
with open(OUTPUT, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=[
        "date", "file", "speech_id", "type", "speaker", "speaker_id",
        "colnum", "time", "url", "section", "text", "word_count"
    ])
    writer.writeheader()

# append each date
for i, date in enumerate(DATES):
    rows = parse(date)
    if rows:
        with open(OUTPUT, "a", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=rows[0].keys())
            writer.writerows(rows)
        print(f"[{i+1}/{len(DATES)}] {date} — {len(rows)} speeches")

[1/11526] 1952-01-29 — 401 speeches
[2/11526] 1952-01-30 — 468 speeches
[3/11526] 1952-01-31 — 493 speeches
[4/11526] 1952-02-01 — 128 speeches
[5/11526] 1952-02-04 — 425 speeches
[6/11526] 1952-02-05 — 465 speeches
[7/11526] 1952-02-06 — 10 speeches
[8/11526] 1952-02-07 — 17 speeches
[9/11526] 1952-02-08 — 32 speeches
[10/11526] 1952-02-11 — 27 speeches
[11/11526] 1952-02-19 — 592 speeches
[12/11526] 1952-02-20 — 519 speeches
[13/11526] 1952-02-21 — 743 speeches
[14/11526] 1952-02-22 — 134 speeches
[15/11526] 1952-02-25 — 719 speeches
[16/11526] 1952-02-26 — 709 speeches
[17/11526] 1952-02-27 — 897 speeches
[18/11526] 1952-02-28 — 502 speeches
[19/11526] 1952-02-29 — 172 speeches
[20/11526] 1952-03-03 — 475 speeches
[21/11526] 1952-03-04 — 519 speeches
[22/11526] 1952-03-05 — 611 speeches
[23/11526] 1952-03-06 — 641 speeches
[24/11526] 1952-03-07 — 117 speeches
[25/11526] 1952-03-10 — 594 speeches
[26/11526] 1952-03-11 — 466 speeches
[27/11526] 1952-03-12 — 514 speeches
[28/11526] 195